In [1]:
import sys
import os
import matplotlib.pyplot as plt
import cv2
import numpy as np

# Add the src directory to the path. TEMPORARY FIX
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../..")))

from src.data_processing.dataset_loader import CoastData

In [4]:
data_path = os.path.abspath(os.path.join(os.getcwd(), "../../../data/SIRENA/"))

# Load the data, all the different stations
data = CoastData(data_path)
station_names = data.get_station_names()

CLASS_MAP = {
    0: "NoData",
    25: "NotClassified",
    75: "Landwards",
    150: "Seawards",
    255: "Shoreline"
}

total_pixels_station_and_class = {station: {cls: 0 for cls in CLASS_MAP.keys()} for station in station_names} 
total_pixels_station = {station: 0 for station in station_names}
total_pixels_global = {cls: 0 for cls in CLASS_MAP.keys()}
print(total_pixels_station_and_class)

station_shapes = {station: {'heights': [], 'widths': []} for station in station_names}

CoastData: global - 2 images
{'snb': {0: 0, 25: 0, 75: 0, 150: 0, 255: 0}}


In [5]:
for station in station_names:
    # print(station)
    station_data = data.get_images(station)
    print(f"\nStation: {station}")
    print(f"Number of images in station {station}: {len(station_data)}")

    for image_data in station_data:

        mask = cv2.imread(image_data['mask'], cv2.IMREAD_GRAYSCALE)
        total_pixels = mask.shape[0] * mask.shape[1]
        total_pixels_station[station] += total_pixels
        
        classes, count = np.unique(mask, return_counts=True)
        for cls, cnt in zip(classes, count):
            percentage = (cnt / total_pixels) * 100
            total_pixels_station_and_class[station][cls] += cnt
            total_pixels_global[cls] += cnt
            
        
        height, width = mask.shape
        station_shapes[station]['heights'].append(height)
        station_shapes[station]['widths'].append(width)

    for cls, cnt in total_pixels_station_and_class[station].items():
        percentage = (cnt / total_pixels_station[station]) * 100 if total_pixels_station[station] > 0 else 0
        print(f"Class {CLASS_MAP[cls]}: {cnt} pixels ({percentage:.2f}%)")

    heights = station_shapes[station]['heights']
    widths = station_shapes[station]['widths']
    print(f"Image heights in station {station}: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.2f}, std={np.std(heights):.2f}")
    print(f"Image widths in station {station}: min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.2f}, std={np.std(widths):.2f}")

print("\nGlobal statistics:")
for cls, cnt in total_pixels_global.items():
    total_pixels = sum(total_pixels_global.values())
    percentage = (cnt / total_pixels) * 100 if total_pixels > 0 else 0
    print(f"Class {CLASS_MAP[cls]}: {cnt} pixels ({percentage:.2f}%)")



Station: snb
Number of images in station snb: 2
Class NoData: 1561777 pixels (20.34%)
Class NotClassified: 2627613 pixels (34.21%)
Class Landwards: 595259 pixels (7.75%)
Class Seawards: 2891453 pixels (37.65%)
Class Shoreline: 3898 pixels (0.05%)
Image heights in station snb: min=1600, max=1600, mean=1600.00, std=0.00
Image widths in station snb: min=2400, max=2400, mean=2400.00, std=0.00

Global statistics:
Class NoData: 1561777 pixels (20.34%)
Class NotClassified: 2627613 pixels (34.21%)
Class Landwards: 595259 pixels (7.75%)
Class Seawards: 2891453 pixels (37.65%)
Class Shoreline: 3898 pixels (0.05%)
